# 03 · Data Enrichment


-We will only focus on movies from 1980 to 2025

-We select the top 10 films for each genre (top 10 most voted)and the 3 most important principals

-By using the wikidata source we can link the IMDB key and the wikipedia url for the film or principal


Requests are serial, rate-limited and identified with a contact address, and every answer is cached, so an interrupted run resumes without asking again.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run the notebook from the IMDb repository or its notebooks folder.")
sys.path.insert(0, str(ROOT))

import csv
import json
import time

from pyspark import StorageLevel
from pyspark.sql import functions as F
from src import enrichment as enrich
from src import wiki_sources as wiki
from src.imdb_loader import data_root, prepared_snapshot, read_prepared, spark_session

DATA = data_root(ROOT)
snapshot = prepared_snapshot(DATA)
spark = spark_session("IMDb Data Enrichment")
tables = read_prepared(spark, snapshot)

PARAMETERS = dict(first_year=1980, last_year=2025, min_minutes=40, top_n=10, principals_per_film=3)
# Number of identifiers to gather per step. Unset (default) gathers everything;
# set GATHER_LIMIT=10 in the environment for a quick first test.
GATHER_LIMIT = int(os.getenv("GATHER_LIMIT", "0")) or None
# Wikimedia's User-Agent policy requires a real contact address.
CONTACT = os.getenv("WIKI_CONTACT", "")
if "@" not in CONTACT:
    raise RuntimeError("Set WIKI_CONTACT to a real e-mail address of the group before running this notebook.")

WORK = DATA / "enrichment"
IDS, RAW = WORK / "ids", WORK / "raw"
OUTPUT = WORK / "output" / snapshot.name
RAW.mkdir(parents=True, exist_ok=True)
print(f"Prepared snapshot: {snapshot.name}")

Prepared snapshot: 55b6a512c5f24516a92f5a05ef0961fa


## Step 1 · Select films and principals (Spark)

We have to select the top ten movies per genre and per year ranking them by the most votes.The functions used are defined in enrichement.py

then within those movies we search for the 3 most significant principals



In [5]:
selected = enrich.select_top_films(
    tables["title.basics"], tables["title.ratings"],
    first_year=PARAMETERS["first_year"], last_year=PARAMETERS["last_year"],
    min_minutes=PARAMETERS["min_minutes"], top_n=PARAMETERS["top_n"],
).persist(StorageLevel.MEMORY_AND_DISK)
chosen = enrich.select_principals(
    tables["title.principals"], selected, per_film=PARAMETERS["principals_per_film"]
).persist(StorageLevel.MEMORY_AND_DISK)

enrich.write_ids(selected.withColumnRenamed("tconst", "imdb_id"), "imdb_id", (IDS / "films").as_posix())
enrich.write_ids(chosen.withColumnRenamed("nconst", "imdb_id"), "imdb_id", (IDS / "people").as_posix())

print(f"Film-genre-year rows : {selected.count():,}")
print(f"Distinct films       : {selected.select('tconst').distinct().count():,}")
print(f"Film principals      : {chosen.count():,}")
print(f"Distinct people      : {chosen.select('nconst').distinct().count():,}")
selected.filter(F.col("startYear") == 2010).orderBy("genre", "genre_rank").show(12, truncate=40)

Film-genre-year rows : 10,198
Distinct films       : 7,186
Film principals      : 21,493
Distinct people      : 11,283
+---------+----------------------------------------+---------+--------------+-------------+--------+------+----------+
|   tconst|                            primaryTitle|startYear|runtimeMinutes|averageRating|numVotes| genre|genre_rank|
+---------+----------------------------------------+---------+--------------+-------------+--------+------+----------+
|tt1228705|                              Iron Man 2|     2010|           124|          6.9|  964035|Action|         1|
|tt0892769|                How to Train Your Dragon|     2010|            98|          8.1|  913819|Action|         2|
|tt1250777|                                Kick-Ass|     2010|           117|          7.6|  626337|Action|         3|
|tt0446029|             Scott Pilgrim vs. the World|     2010|           112|          7.5|  502829|Action|         4|
|tt0964517|                             The Figh

## Step 2 · Gather Wikidata links and Wikipedia text 

we set up the fetching of the data from wikipedia for both selected movies and people.

In [6]:
client = wiki.PoliteClient(CONTACT, min_interval=0.2)
ids = (*wiki.read_ids(IDS / "films", "imdb_id"), *wiki.read_ids(IDS / "people", "imdb_id"))
start = time.monotonic()
added = wiki.fetch_wikidata(client, ids, RAW / "wikidata.jsonl", limit=GATHER_LIMIT)
print(f"Wikidata: {added:,} identifiers added in {time.monotonic() - start:.0f}s")

Wikidata: 10 identifiers added in 1s


In [7]:
start = time.monotonic()
films_added = wiki.fetch_articles(
    client, wiki.article_urls(RAW / "wikidata.jsonl", "tt"), RAW / "wikipedia_films.jsonl",
    intro_only=False, limit=GATHER_LIMIT,
)
people_added = wiki.fetch_articles(
    client, wiki.article_urls(RAW / "wikidata.jsonl", "nm"), RAW / "wikipedia_people.jsonl",
    intro_only=True, limit=GATHER_LIMIT,
)
print(f"Wikipedia: {films_added:,} film and {people_added:,} people articles added "
      f"in {time.monotonic() - start:.0f}s ({client.requests_sent:,} requests this session)")

Wikipedia: 4 film and 1 people articles added in 10s (6 requests this session)


## Step 3 · Clean, join and enforce the schema (Spark)



In [8]:
links = enrich.wikidata_links(
    enrich.read_jsonl(spark, (RAW / "wikidata.jsonl").as_posix(), enrich.WIKIDATA_SCHEMA)
)
film_pages = enrich.article_sections(
    enrich.read_jsonl(spark, (RAW / "wikipedia_films.jsonl").as_posix(), enrich.ARTICLE_SCHEMA)
)
people_pages = enrich.article_sections(
    enrich.read_jsonl(spark, (RAW / "wikipedia_people.jsonl").as_posix(), enrich.ARTICLE_SCHEMA)
)

films = enrich.enrich_films(selected, links, film_pages, chosen,
                            min_principals=PARAMETERS["principals_per_film"])
principals = enrich.enrich_principals(chosen, tables["name.basics"], links, people_pages)
films.write.mode("overwrite").parquet((OUTPUT / "films").as_posix())
principals.write.mode("overwrite").parquet((OUTPUT / "principals").as_posix())

films = spark.read.parquet((OUTPUT / "films").as_posix())
principals = spark.read.parquet((OUTPUT / "principals").as_posix())
assert films.filter(F.col("source_url").isNull()).count() == 0, "Film without lineage"
assert principals.filter(F.col("source_url").isNull()).count() == 0, "Principal without lineage"
assert films.filter((F.col("enrichment_status") == "enriched") & F.col("synopsis").isNull()).count() == 0
print(f"Saved to {OUTPUT}")

Saved to C:\Users\anasb\BigData\Project\data\enrichment\output\55b6a512c5f24516a92f5a05ef0961fa


## Results · Coverage and examples

In [9]:
print("Films")
enrich.coverage(films).show(truncate=80)
print("Principals")
enrich.coverage(principals).show(truncate=80)
print("Films by number of principals listed in IMDb")
films.groupBy("principal_count", "principals_note").count().orderBy("principal_count").show(truncate=80)

films.filter(F.col("enrichment_status") == "enriched").select(
    "tconst", "primaryTitle", "startYear", F.substring("synopsis", 1, 120).alias("synopsis"),
    "source_revision_url",
).orderBy(F.desc("numVotes")).show(5, truncate=60)
principals.filter(F.col("enrichment_status") == "enriched").select(
    "tconst", "primaryName", "category", F.substring("biography", 1, 100).alias("biography"), "source_url",
).show(15, truncate=60)

Films


+-----------------+--------------------------------------------------------------------------------+-----+----+
|enrichment_status|                                                                 enrichment_note|count| pct|
+-----------------+--------------------------------------------------------------------------------+-----+----+
|         enriched|                                                                            NULL| 6000|83.5|
|    not_available|No Wikidata item carries this IMDb identifier (P345); checked via https://que...|  490|6.82|
|    not_available|                               Wikipedia article has no plot or synopsis section|  368|5.12|
|    not_available|                                  Wikidata item has no English Wikipedia article|  328|4.56|
+-----------------+--------------------------------------------------------------------------------+-----+----+

Principals
+-----------------+-------------------------------------------------------------------------

## Bonus · Original languages for exploration Q6

Wikidata's *original language of film* (P364) is exported in the format expected by `02_data_exploration.ipynb` (`data/reference/original_languages.csv`, one row per film and language with its source). Q6 can then run in `verified` mode on the enriched films.

In [10]:
reference = DATA / "reference" / "original_languages.csv"
reference.parent.mkdir(parents=True, exist_ok=True)
languages = films.select(
    "tconst", F.explode("original_languages").alias("language"), F.col("wikidata_uri").alias("source_uri")
).filter(F.col("source_uri").isNotNull())
with reference.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["tconst", "language", "source_uri"])
    rows = 0
    for row in languages.toLocalIterator():
        writer.writerow([row.tconst, row.language, row.source_uri])
        rows += 1
print(f"{rows:,} film-language rows written to {reference}")

7,403 film-language rows written to C:\Users\anasb\BigData\Project\data\reference\original_languages.csv


In [11]:
spark.stop()